In [3]:
from pathlib import Path
from typing import List, Sequence, Optional, Any, Tuple
from plotnine import (
	ggplot,
 	ggtitle,
  	guides, guide_legend,
	aes,
	geom_path, geom_point, geom_text,
	xlab, ylab,
	theme, element_text, element_rect, element_line,
  	scale_y_continuous
   )
from plotnine.themes import theme_classic
import plotnine.scales as scales
import polars as pl
import pandas as pd
import numpy as np

In [4]:
# --- Load Excel file ---
def load_file(file_name: str, sheet_name= None) -> pl.DataFrame:
    
    """Load an Excel file as a DataFrame with a fast engine for processing Excel files.
    Parameters:
    - file_name
    - sheet_name: Excel sheet required or None to load the first sheet
    Returns a pandas DataFrame"""
    
    # 1. Get the folder where THIS script lives
    script_location = Path.cwd()
    
    # 2. Build the path to the file relative to the script
    file_path = script_location / "Data" / file_name
    
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f'File not found: {file_path}')
    # Let pandas choose a working engine (openpyxl, xlrd, etc.). 
    # If your environment requires a specific engine, pass it here.
    df = pl.read_excel(path, sheet_name=sheet_name, engine="calamine")
    df.columns = [
    c.strip()
    .replace(" (", "(")   # <--- Removes space before (
    .replace(" ", "_")    # <--- Turns remaining spaces to underscores
    for c in df.columns
    ]
    return df

In [7]:
# Example:
df = load_file(r"C:\Users\meize\OneDrive\سطح المكتب\Airbin\LFP_500_cycles\Moha_LFP_142_1C_ECEMCVC_2_CHC10_2026_03_13_183407\Moha_LFP_142_1C_ECEMCVC_2_CHC10_Channel_10_Wb_1.xlsx", "Channel-10_1")
df.head()

Could not determine dtype for column 13, falling back to string
Could not determine dtype for column 15, falling back to string


Data_Point,Date_Time,Test_Time(s),Step_Time(s),Cycle_Index,Step_Index,Current(A),Voltage(V),Power(W),Charge_Capacity(Ah),Discharge_Capacity(Ah),Charge_Energy(Wh),Discharge_Energy(Wh),ACR(Ohm),dV/dt(V/s),Internal_Resistance(Ohm),dQ/dV(Ah/V),dV/dQ(V/Ah)
i64,datetime[ms],f64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,str,f64,str,f64,f64
1,2026-03-13 18:03:27.166,60.0002,60.0002,1,1,0.0,2.763,0.0,0.0,0.0,0.0,0.0,null,-0.000014,null,null,null
2,2026-03-13 18:04:27.166,120.0003,120.0003,1,1,0.0,2.7621,0.0,0.0,0.0,0.0,0.0,null,-0.000013,null,null,null
3,2026-03-13 18:05:27.166,180.0002,180.0002,1,1,0.0,2.7613,0.0,0.0,0.0,0.0,0.0,null,-0.000014,null,0.0,0.0
4,2026-03-13 18:06:27.166,240.0001,240.0001,1,1,0.0,2.7605,0.0,0.0,0.0,0.0,0.0,null,-0.000005,null,null,null
5,2026-03-13 18:07:27.166,300.0001,300.0001,1,1,0.0,2.7597,0.0,0.0,0.0,0.0,0.0,null,-0.000018,null,0.0,0.0


In [8]:
def edit_excel(active_m: float, df: pl.DataFrame, export_excel: bool, path: str = None) -> pl.DataFrame:
    
    # Add computed columns in one call (casts for safety)
    df = df.drop("Internal_Resistance(Ohm)", "ACR(Ohm)")
    df = df.with_columns([
        (pl.col("Step_Time(s)").cast(pl.Float64) / 3600).alias("Time(h)"),
        (abs((pl.col("Step_Time(s)").cast(pl.Float64) / 3600) * pl.col("Current(A)").cast(pl.Float64) * 1000 / active_m))
            .alias("Specific_Capacity(mAh/g)"),
        (pl.col("dQ/dV(Ah/V)").cast(pl.Float64) * 1000 / active_m).alias("Specific_dQ/dV")
        ])

    if export_excel:
        # Save: prefer pandas fallback for broad compatibility
        df.to_pandas().to_excel(path if path is not None else "Edited_Excel.xlsx", index=False)
    
    return df
#df = load_file("LFP/LFP_2nd_Test/moha_LFP_grad5C_14mg_Channel_6_Wb_1.xlsx", "Channel-6_1")
#df = edit_excel(0.0031, df, export_excel=False)


In [ ]:
#df.head()

In [9]:
# --- Filtering function ---
def filtering(df: pl.DataFrame, filter_column_names: Sequence[str], ranges: Sequence[Sequence[Any]]) -> pl.DataFrame:
    """Filter `df` by multiple columns using membership tests (isin).
    - `filter_column_names`: sequence of column names to filter on
    - `ranges`: sequence of sequences containing allowed values for each corresponding column
    Returns the filtered DataFrame."""
    if len(filter_column_names) != len(ranges):
        raise ValueError('filter_column_names and ranges must have the same length')
    
    # Use lazy execution for better performance on chains of filters
    lf = df.lazy()
    
    for col, criteria in zip(filter_column_names, ranges):
        if col not in df.columns:
            raise KeyError(f"Column '{col}' not found in DataFrame")
        
        # LOGIC CHANGE: Check type of 'criteria'
        if isinstance(criteria, tuple) and len(criteria) == 2:
            # If it's a tuple (min, max), use is_between
            # accurate for "around 2.5" logic
            lower_bound, upper_bound = criteria
            lf = lf.filter(pl.col(col).is_between(lower_bound, upper_bound))
        else:
            # If it's a list/sequence, use standard is_in
            lf = lf.filter(pl.col(col).is_in(criteria))
        
    return lf.collect()

In [10]:
#Plotting Specific_dQ/dV vs Voltage(V)
def spec_dQdV_df(df: pl.DataFrame, range: Sequence[Sequence[int]]) -> pl.DataFrame:
    """Prepare DataFrame for plotting Specific_dQ/dV vs Voltage(V).
    Filters for Cycle_Index 1 and Step_Index 2 or 3, adjusts Specific_dQ/dV sign for discharge steps.
    Returns the modified DataFrame."""
    
    discharge_steps = range[1][1::2]  # Define discharge steps
    
    df_sp_dQdV = filtering(df, ["Cycle_Index", "Step_Index"], range)

    df_sp_dQdV = df_sp_dQdV.filter(pl.col("Specific_dQ/dV").is_not_null() &
                                (pl.col("Specific_dQ/dV").is_not_nan()) &
                                (pl.col("Specific_dQ/dV") != 0)
                                )
    
    df_sp_dQdV = df_sp_dQdV.with_columns(
                            pl.when((pl.col("Step_Index").is_in(discharge_steps)))
                            .then(pl.col("Specific_dQ/dV") * -1)
                            .otherwise(pl.col("Specific_dQ/dV"))
                            .alias("Specific_dQ/dV")  # Ensures it overwrites the original column
                            )
    
    df_sp_dQdV = df_sp_dQdV.filter(pl.col("dQ/dV(Ah/V)").is_not_null() &
                                (pl.col("dQ/dV(Ah/V)").is_not_nan()) &
                                (pl.col("dQ/dV(Ah/V)") != 0)
                                )
    
    df_sp_dQdV = df_sp_dQdV.with_columns(
                            pl.when((pl.col("Step_Index").is_in(discharge_steps)))
                            .then(pl.col("dQ/dV(Ah/V)") * -1)
                            .otherwise(pl.col("dQ/dV(Ah/V)"))
                            .alias("dQ/dV(Ah/V)")  # Ensures it overwrites the original column
                            )
    return df_sp_dQdV

In [11]:
from plotnine import scale_color_manual


def plotnine_ploting(
    df: pl.DataFrame,
    multiple_graphs: bool = True,
    show_legend: bool = True,
    labels: bool = False,
    points_and_lines: int = 0,
    column_filter: Optional[str] = '',
    color_filter: Optional[str] = '',
    column_x: str = 'Cycle_Index',
    column_y: str = 'Specific Capacity (mAh/g)',
    x_label: str = '',
    y_label: str = '',
    y_continuous: bool = False,
    plot_title: str = 'Plot',
    return_ggplot: bool = False,
    folder_name: str = 'default',
    markevery: int = 1,
    save_base: str = 'Cycling/plots',
    overwrite: bool = False,
    dpi: int = 400,
    width: float = 8,
    height: float = 6,
    units: str = 'in')-> Tuple[str, str]:
    
    # Convert to pandas for plotnine
    df = pd.DataFrame(df.to_pandas())
    # Build ggplot using aes_string for dynamic column names
    # Base ggplot mapping
    aes_mapping = dict(x=column_x, y=column_y)
    
    # Determine if grouping should be applied
    use_grouping = column_filter in df.columns if column_filter else False
    
    # Prepare grouping column if present
    if use_grouping:
        # Preserve first-seen order for column_filter
        first_seen_filter = df[column_filter].drop_duplicates()
        df[column_filter] = pd.Categorical(
            df[column_filter],
            categories=first_seen_filter,
            ordered=True
        )
        
        # Preserve first-seen order for color_filter
        first_seen_color = df[color_filter].drop_duplicates()
        df[color_filter] = pd.Categorical(
            df[color_filter],
            categories=first_seen_color,
            ordered=True
        )
                
    if multiple_graphs and use_grouping:
        aes_mapping.update(color=color_filter, group=column_filter)
    gg = ggplot(df, aes(**aes_mapping))

    # Sample points for markers
    if markevery > 1:
        if multiple_graphs and use_grouping:
            point_df = (df.assign(_idx=df.groupby(column_filter, observed=True).cumcount())
                        .loc[lambda d: d["_idx"] % markevery == 0].drop(columns="_idx"))
        else:
            point_df = df.iloc[::markevery].copy()
    else:
        point_df = df.copy()

    # Define layers
    layers = []
    if points_and_lines in [0, 2]:
        layers.append(geom_path(size=0.8))  # Slightly thicker line for publications
    if points_and_lines in [1, 2]:
        layers.append(geom_point(data=point_df, size=4, stroke=0.5))  # Larger points for visibility
        point_df_round = point_df.copy()
        point_df_round['label_val'] = np.ceil(point_df_round[column_y]).astype(int)
        #print(point_df_round[['label_val', column_y]])  # Debug: Check label values
        if labels:                
            layers.append(
                geom_text(
                    data=point_df_round, 
                    mapping = aes(label='label_val'), 
                    size=7, 
                    va='bottom', 
                    ha='center', 
                    nudge_y=0.02*point_df[column_y].max()
                    )
                )
    for layer in layers:
        gg = gg + layer

    # Colormap for multiple graphs
    if multiple_graphs and use_grouping:
        gg = gg + scales.scale_colour_discrete()
        if color_filter == 'C_rate':  # Custom colors for C rates
            gg += scale_color_manual(
                values={
                    "C/20 start": "#1f77b4",
                    "C/10": "#ff7f0e",
                    "C/5": "#34b434",
                    "C/3": "#d62728",
                    "1C": "#9467bd",
                    "2C": "#1a4d07",
                    "3C": "#e377c2",
                    "4C": "#bcbd22",
                    "5C": "#8c564b",
                    "C/20 end": "#3B3B3B"
                }
            )
    # Add scales safely
    if y_continuous:
        gg += scale_y_continuous(limits=(0, None))

    gg = (gg + ggtitle(plot_title) 
          + xlab(x_label)
          + ylab(y_label)
          + theme_classic(base_size=12)
          + theme(                  
                axis_text=element_text(size=14),
                axis_title=element_text(size=14),
                plot_title=element_text(size=14, weight='bold'),
                legend_position=(0.1, 0.15) if show_legend else 'none',
                legend_direction='horizontal', # make legend items arranged horizontally
                legend_justification=(0,1),
                legend_background=element_rect(
                    fill='white',    # background color
                    color='black',   # border color
                    size=1,          # border thickness
                    linetype='dashed' # solid, dashed, etc.
                ),
                panel_background=element_rect(fill='white'),
                panel_border=element_rect(color='black', size=2),
                figure_size=(width, height),
                panel_grid_major=element_line(color="gray", size=0.7),
                panel_grid_minor=element_line(color="gray", size=0.25),
         )
          + guides(
            color=guide_legend(
                override_aes={'size': 1.5, 'stroke': 1.5}
            )
            )
        )

    if not show_legend:
        gg = gg + theme(legend_position='none')
    
    if return_ggplot:
        return gg
    else:
        gg.show()
    # Prepare save paths
    save_folder = Path(save_base) / folder_name
    save_folder.mkdir(parents=True, exist_ok=True)
    safe_title = plot_title.replace(' ', '_').replace('/', '_')
    save_paths = {
        'PNG': save_folder / f"{safe_title}.png",
        'PDF': save_folder / f"{safe_title}.pdf"
    }

    # Save files
    for fmt, path in save_paths.items():
        if path.exists() and not overwrite:
            print(f"Skipping existing {fmt} (overwrite=False): {path}")
        else:
            gg.save(str(path), dpi=dpi if fmt == 'PNG' else None, width=width, height=height, units=units)
            print(f"{fmt} saved to {path}")

    return str(save_paths['PNG']), str(save_paths['PDF'])

In [12]:
def build_selected_cycles(
    df: pl.DataFrame,
    start: Tuple[int, int],
    mid: int,
    end: Tuple[int, int],
    exclude_range: Tuple[int, int] = None
) -> List[int]:
    cycles = sorted(df['Cycle_Index'].unique())

    mid_cycles = [c for c in cycles if c % mid == 0]

    if exclude_range:
        low, high = exclude_range
        mid_cycles = [c for c in mid_cycles if not (low <= c <= high)]

    return sorted(set(
        cycles[start[0]:start[1]] +
        mid_cycles +
        cycles[end[0]:end[1]]
    ))

In [13]:
file_path = r"C:\Users\meize\OneDrive\سطح المكتب\Airbin\LFP_500_cycles\Moha_LFP_142_1C_ECEMCVC_2_CHC10_2026_03_13_183407\Moha_LFP_142_1C_ECEMCVC_2_CHC10_Channel_10_Wb_1.xlsx"
folder_name = Path(file_path).stem
df = load_file(file_path, "Channel-10_1")
df = df.with_columns(
    pl.when(pl.col("Step_Index").is_in([2,3])).then(pl.lit("C/20"))
    .when(pl.col("Step_Index").is_in([5,6])).then(pl.lit("C/10"))
    .when(pl.col("Step_Index").is_in([8,9])).then(pl.lit("C/5"))
    .when(pl.col("Step_Index").is_in([11,12])).then(pl.lit("C/3"))
    .when(pl.col("Step_Index").is_in([14,15])).then(pl.lit("1C"))
    .when(pl.col("Step_Index").is_in([17,18])).then(pl.lit("5C"))
    .when(pl.col("Step_Index").is_in([20,21])).then(pl.lit("C/20"))
    .alias("C_rate")
)
df = df.with_columns(
    pl.when(pl.col("Current(A)") > 0)
    .then(pl.lit("Charge"))
    .when(pl.col("Current(A)") < 0)
    .then(pl.lit("Discharge"))
    .otherwise(None)
    .alias("Direction")
)
# Combined group column for plotnine
df = df.with_columns([(pl.col("C_rate") + "_" + pl.col("Direction")).alias("C_rate_group")])

step5_df = df.filter(pl.col("Step_Index") == 5)
print(step5_df)

Could not determine dtype for column 13, falling back to string
Could not determine dtype for column 15, falling back to string


shape: (17_239, 21)
┌────────────┬────────────┬───────────┬───────────┬───┬───────────┬────────┬───────────┬───────────┐
│ Data_Point ┆ Date_Time  ┆ Test_Time ┆ Step_Time ┆ … ┆ dV/dQ(V/A ┆ C_rate ┆ Direction ┆ C_rate_gr │
│ ---        ┆ ---        ┆ (s)       ┆ (s)       ┆   ┆ h)        ┆ ---    ┆ ---       ┆ oup       │
│ i64        ┆ datetime[m ┆ ---       ┆ ---       ┆   ┆ ---       ┆ str    ┆ str       ┆ ---       │
│            ┆ s]         ┆ f64       ┆ f64       ┆   ┆ f64       ┆        ┆           ┆ str       │
╞════════════╪════════════╪═══════════╪═══════════╪═══╪═══════════╪════════╪═══════════╪═══════════╡
│ 40321      ┆ 2026-03-18 ┆ 386482.05 ┆ 0.0052    ┆ … ┆ null      ┆ C/10   ┆ Charge    ┆ C/10_Char │
│            ┆ 05:23:49.2 ┆ 22        ┆           ┆   ┆           ┆        ┆           ┆ ge        │
│            ┆ 18         ┆           ┆           ┆   ┆           ┆        ┆           ┆           │
│ 40322      ┆ 2026-03-18 ┆ 386482.06 ┆ 0.0177    ┆ … ┆ null      ┆ C/1

In [ ]:

# df_spec_dQdV = 
# spec_dQdV_df(df, [[1, 2],[2,3, 5,6, 8,9, 11,12]])
# [3, 6, 9, 12]
# [2, 5, 8, 11]

# df_spec_dQdV = spec_dQdV_df(df, [[1, 2],[2,3, 6,7, 10,11, 14,15]])
# [3, 7, 11, 15]
# [2, 6, 10, 14]

file_path = r"C:\Users\meize\OneDrive\سطح المكتب\Airbin\LFP_500_cycles\Moha_LFP_142_1C_ECEMCVC_2_CHC10_2026_03_13_183407\Moha_LFP_142_1C_ECEMCVC_2_CHC10_Channel_10_Wb_1.xlsx"
folder_name = Path(file_path).stem
df = load_file(file_path, "Channel-10_1")
df = df.with_columns(
    pl.when(pl.col("Step_Index").is_in([2,3])).then(pl.lit("C/20 start"))
    .when(pl.col("Step_Index").is_in([5,6])).then(pl.lit("C/10"))
    .when(pl.col("Step_Index").is_in([8,9])).then(pl.lit("C/5"))
    .when(pl.col("Step_Index").is_in([11,12])).then(pl.lit("C/3"))
    .when(pl.col("Step_Index").is_in([14,15])).then(pl.lit("1C"))
    .when(pl.col("Step_Index").is_in([17,18])).then(pl.lit("C/20 end"))
    .otherwise(pl.lit(None))
    .alias("C_rate")
)
"""df = df.with_columns(
    pl.when(pl.col("Step_Index").is_in([1,2])).then(pl.lit("1C"))
    .when(pl.col("Step_Index").is_in([4,5])).then(pl.lit("2C"))
    .when(pl.col("Step_Index").is_in([7,8])).then(pl.lit("3C"))
    .when(pl.col("Step_Index").is_in([10,11])).then(pl.lit("4C"))
    .when(pl.col("Step_Index").is_in([13,14])).then(pl.lit("C/20 end"))
    .otherwise(pl.lit(None))
    .alias("C_rate")
)"""
df = df.with_columns(
    pl.when(pl.col("Current(A)") > 0)
    .then(pl.lit("Charge"))
    .when(pl.col("Current(A)") < 0)
    .then(pl.lit("Discharge"))
    .otherwise(None)
    .alias("Direction")
)
# Combined group column for plotnine
df = df.with_columns([(pl.col("Cycle_Index").cast(pl.Utf8) + "_" + pl.col("Direction")).alias("C_rate_group")])



df = edit_excel(0.010081, df, export_excel=False)
#df_spec_dQdV = spec_dQdV_df(df, [[2, 5, 8, 11, 13],[1,2, 4,5, 7,8, 10,11, 13,14]])
#df_spec_dQdV = spec_dQdV_df(df, [[2, 15, 20],[1,2, 4,5, 7,8, 10,11, 13,14]])
df_spec_dQdV = spec_dQdV_df(df, [[2, 4, 7, 10, 20, 80, 100, 150, 200, 300, 500,512, 513],[2,3, 5,6, 8,9, 11,12, 14,15, 17,18]])


selected_cycles = build_selected_cycles(df, start=(0, 12), mid=100, end=(-3, -1), exclude_range=(100, 500))
print('Selected cycles:', selected_cycles)

# 1. Discharge retention (example)
#filtered_df = filtering(df, ['Step_Index', 'Voltage(V)', 'Cycle_Index'], [[2, 5, 8, 11, 14], (2.3, 2.501), selected_cycles])
filtered_df = filtering(df, ['Step_Index', 'Voltage(V)', 'Cycle_Index'], [[3, 6, 9, 12, 15, 18], (2.3, 2.501), selected_cycles])
plotnine_ploting(filtered_df, labels=True, column_filter='Step_Index', color_filter='C_rate',
                 points_and_lines=1, 
                 column_x='Cycle_Index', column_y='Specific_Capacity(mAh/g)',
                 x_label='Cycle index', y_label='Discharge specific capacity (mAh/g)',
                 y_continuous=True,
                 plot_title='Discharge Retention', folder_name=folder_name, 
                 markevery=1, save_base='Cycling/plots', overwrite=True)

# 2. Charge retention (example)
#filtered_df = filtering(df, ['Step_Index', 'Voltage(V)', 'Cycle_Index'], [[1, 4, 7, 10, 13], (4.199, 4.4), selected_cycles])
filtered_df = filtering(df, ['Step_Index', 'Voltage(V)', 'Cycle_Index'], [[2, 5, 8, 11, 14, 17], (4.199, 4.4), selected_cycles])

plotnine_ploting(filtered_df, labels=True, column_filter='Step_Index', color_filter='C_rate', 
                 points_and_lines=1,
                 column_x='Cycle_Index', column_y='Specific_Capacity(mAh/g)',
                 x_label='Cycle index', y_label='Charge specific capacity (mAh/g)', 
                 y_continuous=True,
                 plot_title='Charge Retention', folder_name=folder_name, 
                 markevery=1,save_base='Cycling/plots', overwrite=True)

selected_cycles = build_selected_cycles(df, start=(0, 12), mid=100, end=(-3, -1), exclude_range=(100, 500))

# 3. Charging Voltage vs Specific Capacity
#filtered_df = filtering(df, ['Step_Index', 'Cycle_Index'], [[1,2, 4,5, 7,8, 10,11, 13,14], selected_cycles])
filtered_df = filtering(df, ['Step_Index', 'Cycle_Index'], [[2,3, 5,6, 8,9, 11,12, 14,15, 17,18], selected_cycles])

plotnine_ploting(filtered_df, multiple_graphs=True, show_legend=True,  
                 points_and_lines=0, column_filter='C_rate_group', color_filter='C_rate', 
                 column_x='Specific_Capacity(mAh/g)', column_y='Voltage(V)',
                 x_label='Specific capacity (mAh/g)', y_label='Voltage (V)',
                 plot_title='Charge/Discharge Voltage (V) vs Specific Capacity (mAh/g)', folder_name=folder_name, 
                 save_base='Cycling/plots', overwrite=True)

# 3. Charging Voltage vs Specific Capacity
#filtered_df = filtering(df, ['Step_Index', 'Cycle_Index'], [[1, 4, 7, 10, 13], selected_cycles])
filtered_df = filtering(df, ['Step_Index', 'Cycle_Index'], [[2, 5, 8, 11, 14, 17], selected_cycles])
plotnine_ploting(filtered_df, multiple_graphs=True, show_legend=True,  
                 points_and_lines=0, column_filter='Cycle_Index', color_filter='C_rate', 
                 column_x='Specific_Capacity(mAh/g)', column_y='Voltage(V)',
                 x_label='Specific capacity (mAh/g)', y_label='Charge voltage (V)',
                 plot_title='Charging Voltage (V) vs Specific Capacity (mAh/g)', folder_name=folder_name, 
                 save_base='Cycling/plots', overwrite=True)

# 4. Discharging Voltage vs Specific Capacity
#filtered_df = filtering(df, ['Step_Index', 'Cycle_Index'], [[2, 5, 8, 11, 14], selected_cycles])
filtered_df = filtering(df, ['Step_Index', 'Cycle_Index'], [[3, 6, 9, 12, 15, 18], selected_cycles])
plotnine_ploting(filtered_df, multiple_graphs=True, show_legend=True, 
                 points_and_lines=0, column_filter='Cycle_Index', color_filter='C_rate', 
                 column_x='Specific_Capacity(mAh/g)', column_y='Voltage(V)', 
                 x_label='Specific capacity (mAh/g)', y_label='Discharge voltage (V)',
                 plot_title='Discharging Voltage (V) vs Specific Capacity (mAh/g)', folder_name=folder_name, 
                 save_base='Cycling/plots', overwrite=True)

plotnine_ploting(df_spec_dQdV, multiple_graphs=True, show_legend=True, 
                 points_and_lines=0, column_filter='Cycle_Index', color_filter='C_rate', 
                 column_x='Voltage(V)', column_y='Specific_dQ/dV', 
                 x_label='Voltage (V)', y_label='Specific dQ/dV',
                 plot_title='Voltage (V) vs Specific dQ/dV', folder_name=folder_name, 
                 save_base='Cycling/plots', overwrite=True)

Could not determine dtype for column 13, falling back to string
Could not determine dtype for column 15, falling back to string


Selected cycles: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 512, 513]


ModuleNotFoundError: No module named 'pyarrow'